In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer


train_dataset = load_dataset("json", data_files="/home/huangruijun/grpo/data/measdata.jsonl",split="train")

# train_dataset, test_dataset = load_dataset(path = "/home/huangruijun/grpo/data/measdata.json")
# train_dataset = train_dataset.shuffle(seed=42)
len(train_dataset)
tokenizer = AutoTokenizer.from_pretrained( "/home/huangruijun/LLaMA-Factory/saves/qwen2-7b-instruct/lora_merge/6stage_meas_50step")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/home/huangruijun/miniconda3/envs/rl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def generate_r1_prompt(query,ans):
    r1_prefix = [{
        "role": "system",
        "content": "You are a helpful assistant."
        },
        { 
        "role": "user",
        "content": query
        }]
    return {"prompt": tokenizer.apply_chat_template(r1_prefix, tokenize=False, continue_final_message=True), "ans": ans}

# convert our dataset to the r1 prompt
train_dataset = train_dataset.map(lambda x: generate_r1_prompt(x["query"],x["ans"]))
train_dataset[0]

Map: 100%|██████████| 298/298 [00:00<00:00, 5071.44 examples/s]


{'docId': 'S0022459611006116-547',
 'query': 'Please help me extract all of the Quantities, MeasuredEntities, MeasuredProperties, and Qualifiers, along with their relationships in the text.\n\nInput: Refined crystallographic parameters for (1) from PXD and PND at 298 K.',
 'ans': 'docId\tannotSet\tannotType\tstartOffset\tendOffset\tannotId\ttext\tother\nS0022459611006116-547\t1\tQuantity\t64\t70\tT1-1\t298 K.\t{"unit": "K"}\nS0022459611006116-547\t1\tMeasuredEntity\t0\t60\tT2-1\tRefined crystallographic parameters for (1) from PXD and PND\t{"HasQuantity": "T1-1"}\n',
 'prompt': '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nPlease help me extract all of the Quantities, MeasuredEntities, MeasuredProperties, and Qualifiers, along with their relationships in the text.\n\nInput: Refined crystallographic parameters for (1) from PXD and PND at 298 K.'}

In [8]:
from quantulum3 import parser

quants = parser.parse('Mg(ClO4)2')
quants[0]

Quantity(2, "Unit(name="dimensionless", entity=Entity("dimensionless"), uri=Dimensionless_quantity)")

In [14]:
from CQE import CQE

parser = CQE.CQE()
text = " the quantity is = 3%"
result = parser.parse(text)
print(result[0])
print(result[0].get_char_indices())

(=,3.0,[%],percentage,{0: [ , quantity]})
{'value': [(19, 20)], 'unit': [(20, 21)]}


In [3]:
import torch
print(torch.cuda.device_count())
print(torch.cuda.is_available())

8
True
